# Project brief · Data Modeling Project
*ERD, normalization, and a star schema with dbt, on a B2B sales dataset*

## 1. Background
A mid-sized company that sells products to business customers around the world keeps its operational data in several systems.
An export of those systems was delivered as one Excel workbook (`dataset_4NF_5NF_practice.xlsx`).
The workbook is **synthetic** (fake data created for practice).
Nobody has documented or validated it, and no reliable model exists.

## 2. Goal
Turn the workbook into a documented, tested, and reproducible data platform:
1. Build the project, the two environments, and the load of the source data.
2. Understand and profile the source data.
3. Describe it with an **ERD**.
4. **Normalize** it into a trusted `core` schema, built with dbt.
5. Build a **dimensional model** (star schema) in `dwh`, built and loaded with dbt.
6. Answer the business questions with SQL.
7. Build a **Power BI** report on top of `dwh`.
8. **Release** the result, including the report, from development to production.

**Nothing is provided except these notebooks and the workbook.** You create every file yourself: repository, Docker setup, scripts, dbt project, SQL, and documents.

## 3. Business questions
The business has asked:

| # | Question |
|---|----------|
| Q1 | How do monthly sales compare with the targets? |
| Q2 | Which product categories and customer segments are the most profitable? |
| Q3 | How fast do orders get shipped and delivered, by shipping method and region? |
| Q4 | How long do customers take to pay, and which invoices are still open? |
| Q5 | Did the marketing campaigns pay off? |
| Q6 | Do we hold enough stock for the products we sell? |
| Q7 | Which customers and regions drive the business? |
| Q8 | Can we deliver each product to each region, and from which suppliers? |
| Q9 | Do we serve customers in their preferred language and shipping method? |

The exact metric definitions are **not** provided. You will agree them with the reviewer in Part 3.

## 4. Required technology
| Tool | Role in this project |
|------|----------------------|
| PostgreSQL 16 in Docker | The two servers: development and production |
| Python (pandas, SQLAlchemy, or similar) | Loads the Excel sheets into `raw`, runs the pipeline, powers these notebooks |
| dbt (core + the PostgreSQL adapter) | Builds `core` and `dwh` from SQL models, with tests and documentation |
| Power BI Desktop (free) | The semantic model, measures, and report on top of `dwh` |
| Jupyter | The tickets and your workspace |
| Git | Version control, one commit per ticket at least |
| An ERD tool of your choice | Chosen and practiced in ticket DM-100 |

## 5. Environments and layers
| Environment | Suggested container | Host port | Suggested database | Purpose |
|-------------|---------------------|-----------|--------------------|---------|
| Development | `dm-pg-dev` | 5433 | `dm_dev` | Build, test, and break things |
| Production | `dm-pg-prod` | 5434 | `dm_prod` | The approved result. **Protected (read-only) until Part 7.** |

Each database has these schemas:

| Schema | Content | Built by |
|--------|---------|----------|
| `raw` | The source data exactly as received | Your Python loader |
| `core` | The cleaned, normalized, trusted model | dbt only |
| `dwh` | The star schema for analytics (the source for Power BI) | dbt only |
| `audit` | Migration tracking, run logs | SQL migrations and your runner |

## 6. Standards
- Names are `snake_case`. In `dwh`, tables use the prefixes `dim_` and `fact_`.
- The `raw` and `audit` structures change only through numbered SQL migration files (for example `V001__create_schemas.sql`).
  The `core` and `dwh` layers are built only by dbt models. No manual changes to the database.
- Production is protected until the release ticket (DM-703).
- Secrets are never committed to Git.
- ERDs are exported to `docs/erd/`, together with the source file of the tool you use.
- Every claim about the data is backed by a query.

## 7. Suggested folder layout
```
data-modeling-project/
├── data/            the source workbook
├── docs/            findings, decisions, dictionaries, reports
│   └── erd/         ERD exports and source files
├── sql/
│   ├── migrations/  numbered structure changes for raw and audit
│   ├── checks/      ad-hoc data quality checks
│   └── queries/     analytical queries
├── dbt/             the dbt project
├── powerbi/         the report and its documentation files
├── notebooks/       these eight notebooks
└── src/             your Python helpers
```
You may choose a different layout if you record why in `docs/decisions.md`.

## 8. Working agreement
- Work on **one ticket at a time**, in order.
- Each ticket ends with a **checkpoint**. Share your work for review before you start the next ticket.
- A ticket is **Done** only when its acceptance criteria are met and the reviewer has signed it off.
- Commit your work after each ticket.

## 9. Log formats
**`docs/findings_log.md`** is a table with these columns: ID, table(s), description, evidence query, rows affected, impact, suggested handling, status.

**`docs/decisions.md`** has one entry per decision, with: date, ticket, context, options considered, decision, reason, consequences, approved by.

# Part 0 · Environment, ingestion, and profiling

**Objective.** Build the project from nothing: repository, two database servers, production protection, connection code, migrations, dbt, the raw load, and a profile of the data.

**Before you start.** Docker, Python 3.10+, Git, and the source workbook are on your computer. Read the project brief first.

**Tickets in this part**

| Ticket | Title | Type | Estimate |
|---|---|---|---|
| DM-001 | Create the project skeleton | Setup | 0.5 day |
| DM-002 | Write the Docker setup for two servers | Setup | 1 day |
| DM-003 | Protect production | Engineering | 1 day |
| DM-004 | Write the connection helper | Engineering | 0.5 day |
| DM-005 | Build a migration runner | Engineering | 1 day |
| DM-006 | Load the source workbook into `raw` | Ingestion | 1 day |
| DM-007 | Profile the data | Analysis | 1 day |
| DM-008 | Set up dbt | Setup | 1 day |

**Ground rules**
- Work on one ticket at a time, in order. Stop at every checkpoint.
- Back every claim about the data with a query.
- Structure changes go in numbered migrations (raw, audit) or in dbt models (core, dwh) only.
- Update your findings log and decisions log as you go.
- Project brief: the first cells of notebook `00_setup_and_raw_load`.

## DM-001 · Create the project skeleton
**Type:** Setup · **Priority:** High · **Estimate:** 0.5 day

**Context**  
Every real project starts with a repository and a shared structure. Nothing exists yet: you create all of it.

**What to do**
- Create a Git repository and the folder layout you will use (the brief suggests one).
- Set up a Python virtual environment and a `requirements.txt`. Add packages as you need them.
- Add a `.gitignore` that keeps secrets, the virtual environment, and generated files out of Git.
- Write a first `README.md` (what the project is and how to start it). You will extend it later.
- Create `docs/findings_log.md` and `docs/decisions.md` using the formats in the brief.
- Create a simple ticket board (a file or a tool) with one row per ticket in these notebooks.

**Deliverables**
- The repository with a first commit
- Folder layout, `.gitignore`, `requirements.txt`, `README.md`
- Both logs and the ticket board

**Acceptance criteria**
- [ ] A new person can clone the repository and see where everything belongs
- [ ] No secret or generated file can be committed by accident
- [ ] Both logs exist and follow the formats in the brief
- [ ] Every ticket in the notebooks appears on the board

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-001.** Share your work for review before you start the next ticket.

## DM-002 · Write the Docker setup for two servers
**Type:** Setup · **Priority:** High · **Estimate:** 1 day

**Context**  
The team works with two servers: development (where we build and break things) and production (where the approved result lives). Both are PostgreSQL 16 in Docker.

**What to do**
- Write a Docker Compose file with two PostgreSQL 16 services. They must differ in container name, host port, database name, user, and data volume (see the environments table in the brief).
- Keep all passwords out of the Compose file and out of Git: read them from an env file that is not committed, and commit an example file with fake values.
- Add health checks, and make the data survive a restart.
- Start both servers and connect to each from a command-line client and from Python.

**Deliverables**
- `docker-compose.yml`
- An env example file (and your private env file, not committed)
- A note in `docs/decisions.md` for every default you changed

**Acceptance criteria**
- [ ] Both servers start from one command and report healthy
- [ ] You can state the port, database, and user that belong to each environment
- [ ] You can prove that a change made in development is not visible in production
- [ ] You can wipe and recreate both servers from scratch with documented commands
- [ ] No password appears in any committed file

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-002.** Share your work for review before you start the next ticket.

## DM-003 · Protect production
**Type:** Engineering · **Priority:** High · **Estimate:** 1 day

**Context**  
Production must not change by accident. Until the release in Part 7, nobody and nothing should be able to write to it by mistake, not even you.

**What to do**
- Research at least two ways to make a PostgreSQL database read-only by default. Compare them (what they block, how easy they are to bypass, how you check their state).
- Choose one, implement it so that it is applied automatically whenever the production server is created, and record the reason.
- Test it: attempt a write in development and in production, and show the results.
- Document how to lift the protection deliberately for a release, how to restore it, and how to check its state at any moment.

**Deliverables**
- The implementation (as code or configuration that runs at creation time)
- Test evidence saved in the notebook
- Decision, lift procedure, and restore procedure in `docs/decisions.md`

**Acceptance criteria**
- [ ] A write to production fails, and the same write to development succeeds
- [ ] The protection is applied automatically on a freshly created production server
- [ ] Reading from production still works
- [ ] You can show the current state of the protection with a query
- [ ] You can explain how it could be bypassed on purpose, and why a real team would add stronger controls (for example separate roles)

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-003.** Share your work for review before you start the next ticket.

## DM-004 · Write the connection helper
**Type:** Engineering · **Priority:** High · **Estimate:** 0.5 day

**Context**  
Notebooks and scripts should not repeat connection details or passwords. One small, safe module does it for everyone.

**What to do**
- Write a Python module that returns a database connection or engine for the environment you ask for (development or production).
- Read settings from the environment, with no hard-coded password anywhere.
- Make production read-only by default, with an explicit option to allow writes. Writing to production should never happen by accident in code either.
- Add a function that confirms which server, database, and user you are connected to.

**Deliverables**
- The module in your source folder
- Evidence in this notebook that it works for both environments

**Acceptance criteria**
- [ ] One line of code gives a development connection, and one gives a production connection
- [ ] A write through the default production connection is refused
- [ ] No secret is stored in code or in Git
- [ ] You can explain what happens if the environment name is wrong or a setting is missing

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-004.** Share your work for review before you start the next ticket.

## DM-005 · Build a migration runner
**Type:** Engineering · **Priority:** High · **Estimate:** 1 day

**Context**  
Changes to the `raw` and `audit` layers must be numbered SQL files, so any environment can be rebuilt from scratch. The same files are applied to development first and production later. The `core` and `dwh` layers are built by dbt (Parts 2 to 4).

**What to do**
- Write the first migration file that creates the `raw` and `audit` schemas (dbt will create `core` and `dwh` itself).
- Build a small runner that applies pending migration files in order and records each one in a tracking table inside `audit`.
- Make the runner take the environment as a parameter.

**Deliverables**
- The first migration file in your migrations folder
- The runner in your source folder

**Acceptance criteria**
- [ ] Running the runner twice changes nothing the second time
- [ ] The tracking table shows file name, checksum, and time applied
- [ ] A failing migration leaves no half-applied change
- [ ] The runner refuses to write to a protected production unless you deliberately allow it

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-005.** Share your work for review before you start the next ticket.

## DM-006 · Load the source workbook into `raw`
**Type:** Ingestion · **Priority:** High · **Estimate:** 1 day

**Context**  
`raw` keeps the source data exactly as received, so that any later problem can be traced back to its origin.

**What to do**
- Read every sheet of the source workbook (it goes in your `data/` folder).
- Load each sheet into its own table in the development `raw` schema, with snake_case names derived from the sheet names.
- Add load metadata columns (source sheet and load timestamp).
- Decide how data types are stored in `raw` and write down why.

**Deliverables**
- Loading code in this notebook (or in your source folder)
- Decision recorded in `docs/decisions.md`

**Acceptance criteria**
- [ ] One raw table per sheet, with row counts equal to the sheet row counts
- [ ] No value was cleaned, renamed, filtered, or converted beyond what loading requires
- [ ] The load can be re-run without duplicating rows
- [ ] Your type decisions are written down with a reason

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-006.** Share your work for review before you start the next ticket.

## DM-007 · Profile the data
**Type:** Analysis · **Priority:** High · **Estimate:** 1 day

**Context**  
Nobody has documented or validated this data. Before modeling anything you must know what you have.

**What to do**
- For every raw table, produce by code: row count, column names and types, null counts, distinct counts, min and max where meaningful, and sample values.
- Examine data quality across the standard dimensions: completeness, uniqueness, validity, consistency, and integrity between tables.
- Record every observation in `docs/findings_log.md` together with the query that proves it.

**Deliverables**
- A rerunnable profiling report
- `docs/findings_log.md` filled in

**Acceptance criteria**
- [ ] The profile covers 100% of raw tables and columns and is generated by code
- [ ] Each finding has an ID, the table(s) involved, a description, an evidence query, the number of rows affected, and a suggested handling (it may be "ask the business")
- [ ] Nothing is fixed at this stage: `raw` stays untouched
- [ ] Findings are ranked by impact

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-007.** Share your work for review before you start the next ticket.

## DM-008 · Set up dbt
**Type:** Setup · **Priority:** High · **Estimate:** 1 day

**Context**  
From Part 2 onward, the transformation layers are built with dbt: SQL models that run inside the database. Python loads the data, and dbt transforms it.

**What to do**
- Install dbt with the PostgreSQL adapter and create the dbt project in a `dbt/` folder.
- Configure a development target and a production target. Credentials must come from the environment, never from a committed file.
- Make sure models land in schemas named exactly `core` and `dwh` (dbt's default naming adds a prefix, so find out how to change that).
- Declare every raw table as a dbt source.
- Create a trivial test model, run it in development, check which schema it lands in, then remove it.
- Generate the dbt documentation site and open it.
- Confirm that a build against production fails while production is protected.

**Deliverables**
- The dbt project with a sources file
- Notes in `docs/decisions.md` on how you structure the project (folders, naming, materializations)

**Acceptance criteria**
- [ ] `dbt debug` passes for the development target
- [ ] All raw tables are declared as sources
- [ ] Models land in `core` and `dwh`, with no prefixes
- [ ] You can explain in your own words what `source()` and `ref()` do and why they matter
- [ ] A production build is refused while production is protected, and nothing changed in production

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-008.** Share your work for review before you start the next ticket.